# 02 · Generate Clay v1.5 embeddings

Loads the chips saved by notebook 01, runs them through the Clay v1.5 foundation model encoder on GPU, and saves one embedding vector per chip.

**Requires a GPU runtime** (Runtime -> Change runtime type -> T4 GPU) — Clay is a ViT-scale transformer, CPU inference over ~1000 chips would be painfully slow.

Clay is *metadata-conditioned*: it doesn't just look at pixels, it's told which physical wavelengths those pixel bands correspond to, and roughly when/where on Earth the image was taken. That's worth sitting with for a second — the embedding for an identical-looking patch of grass could differ depending on whether the model is told it's July or January, Colorado or Chile. Part of what notebook 03 investigates is how much that conditioning actually shows up in the resulting vectors vs. how much is driven by the pixels themselves.

In [ ]:
REPO_URL = "https://github.com/<your-username>/front-range-embeddings.git"  # TODO: fill in

import os

if not os.path.exists("front-range-embeddings"):
    !git clone {REPO_URL} front-range-embeddings
%cd front-range-embeddings
!pip install -q -r environment/requirements-colab.txt

In [ ]:
import sys

sys.path.append(os.getcwd())

import json

import numpy as np
import pandas as pd
import torch

from src import clay_embed

assert torch.cuda.is_available(), "No GPU detected -- switch runtime type to T4 GPU and re-run"
device = "cuda"

chip_pixels = np.load("data/chip_pixels.npy")
with open("data/chips_meta.json") as f:
    chips_meta = json.load(f)

print(f"Loaded {chip_pixels.shape[0]} chips of shape {chip_pixels.shape[1:]}")

In [ ]:
ckpt_path = clay_embed.download_checkpoint()
model = clay_embed.load_model(ckpt_path, device=device)
wavelengths, band_means, band_stds = clay_embed.load_band_stats()

print(f"Loaded Clay v1.5. Wavelengths (nm): {wavelengths.tolist()}")

## Batch-encode all chips

Runs in batches to keep GPU memory bounded. On a free-tier T4 this is on the order of a few minutes for ~1000 chips.

In [ ]:
BATCH_SIZE = 32
dates = [pd.Timestamp(m["date"]) for m in chips_meta]
lats = [m["lat"] for m in chips_meta]
lons = [m["lon"] for m in chips_meta]

all_embeddings = []
n = chip_pixels.shape[0]

for start in range(0, n, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n)
    batch_pixels = clay_embed.normalize_chips(chip_pixels[start:end], band_means, band_stds)
    time_feats, latlon_feats = clay_embed.make_time_latlon_tensors(
        dates[start:end], lats[start:end], lons[start:end]
    )
    batch_emb = clay_embed.encode_batch(
        model, batch_pixels, time_feats, latlon_feats, wavelengths, device=device
    )
    all_embeddings.append(batch_emb)
    if (start // BATCH_SIZE) % 5 == 0:
        print(f"  encoded {end}/{n}")

embeddings = np.concatenate(all_embeddings, axis=0)
print(f"Final embeddings: {embeddings.shape}")

In [ ]:
np.save("data/embeddings.npy", embeddings)
print("Saved data/embeddings.npy")
print("Next: run notebooks/03_analyze_and_export.ipynb")